In [1]:
import ollama

response = ollama.embed(
    model="nomic-embed-text",
    input=["North Korea used cryptocurrency to launder stolen funds from cyber attack.",
    "Hackers converted stolen banking proceeds through digital currencies.",
    ]
)

embedding = response["embeddings"][0]

print(type(embedding))
print(len(embedding))
print(len(response["embeddings"][0]))
print(len(response["embeddings"][1]))

<class 'list'>
768
768
768


In [2]:
import sqlite3
from statistics import mean, median

from osint_agent.models.document import Document
from osint_agent.preprocessing.chunking import chunk_document
from osint_agent.config import settings


conn = sqlite3.connect(settings.DB_PATH)
conn.row_factory = sqlite3.Row

rows = conn.execute(
    f"""
    SELECT *
    FROM documents
    WHERE provider = 'currents'
    ORDER BY retrieved_at DESC
    LIMIT 20
    """
).fetchall()

conn.close()

results = []

for row in rows:
    document = Document(
        doc_id=row["doc_id"],
        title=row["title"],
        source=row["source"],
        provider=row["provider"],
        source_type=row["source_type"],
        published_date=row["published_date"],
        url=row["url"],
        raw_text=row["raw_text"],
        text=row["cleaned_text"],
    )

    chunks = chunk_document(document)

    results.append(
        {
            "doc_id": document.doc_id,
            "title": document.title,
            "text_chars": len(document.text),
            "chunks": chunks,
        }
    )


# ---- Corpus-level summary ----

chunk_counts = [len(result["chunks"]) for result in results]

print(f"Documents: {len(results)}")
print(f"Total chunks: {sum(chunk_counts)}")
print(f"Average chunks/document: {mean(chunk_counts):.2f}")
print(f"Median chunks/document: {median(chunk_counts):.1f}")
print(f"Min chunks/document: {min(chunk_counts)}")
print(f"Max chunks/document: {max(chunk_counts)}")


# ---- Per-document summary ----

print("\nDOCUMENT DISTRIBUTION\n")

for result in results:
    print(
        f"{len(result['chunks']):>2} chunks | "
        f"{result['text_chars']:>6} chars | "
        f"{result['title']}"
    )

/Users/duk3y6/Desktop/aia-osint-briefer/ai-osint-briefing-agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Documents: 20
Total chunks: 20
Average chunks/document: 1.00
Median chunks/document: 1.0
Min chunks/document: 1
Max chunks/document: 1

DOCUMENT DISTRIBUTION

 1 chunks |     83 chars | US imports of Saudi crude fall to zero for first time since 1985 amid Iran conflict
 1 chunks |     94 chars | Joint agreement: Saudi Arabia, Turkey and Pakistan to sign defence pact amid regional violence
 1 chunks |    339 chars | Trump to host mining CEOs as administration seeks minerals for defense supply chains
 1 chunks |    421 chars | Watch Iran Seeks to Bar US, Israeli Ships from Hormuz; OpenAI's Doughnut-Like Device | The Pulse 8/7/2026 - Bloomberg
 1 chunks |    245 chars | South Korea FA 'provided "sexual entertainment" for referees'
 1 chunks |    239 chars | Why Pakistan, Saudi Arabia, Turkey are signing a new defence pact
 1 chunks |    388 chars | Trump to host mining CEOs as administration seeks minerals for defense supply chains
 1 chunks |    385 chars | [Today’s Signal] Hormuz Risk I

In [3]:
sorted_results = sorted(
    results,
    key=lambda result: len(result["chunks"]),
)

sample_results = [
    sorted_results[0],
    sorted_results[len(sorted_results) // 2],
    sorted_results[-1],
]

for result in sample_results:

    print("\n" + "=" * 100)
    print(result["title"])
    print(f"Document ID: {result['doc_id']}")
    print(f"Chunks: {len(result['chunks'])}")
    print("=" * 100)

    for chunk in result["chunks"]:

        print(
            f"\n--- CHUNK {chunk.chunk_index} "
            f"| tokens={chunk.token_count} "
            f"| {chunk.chunk_id} ---\n"
        )

        print(chunk.text)


US imports of Saudi crude fall to zero for first time since 1985 amid Iran conflict
Document ID: currents-5586fdb4-6c4b-5490-a8a1-d8466cef8b9c
Chunks: 1

--- CHUNK 0 | tokens=16 | currents-5586fdb4-6c4b-5490-a8a1-d8466cef8b9c::chunk-000::61ceb22f38f9 ---

us imports of saudi crude fall to zero for first time since 1985 amid iran conflict

Saudi Arabia, Turkey and Pakistan set to sign trilateral defence pact
Document ID: currents-e22f3284-0c4f-5d39-a4bc-4aec5446f795
Chunks: 1

--- CHUNK 0 | tokens=48 | currents-e22f3284-0c4f-5d39-a4bc-4aec5446f795::chunk-000::4933e64dd820 ---

saudi arabia, turkey and pakistan set to sign trilateral defence pact saudi arabia, turkey and pakistan are set to sign a defence agreement in jeddah on friday. the pact could widen military coordination as gulf tensions and regional conflicts intensify.

Houthi missile and drone attacks kill dozens of Yemeni government troops
Document ID: currents-473ab586-3a21-5adc-83f0-393ececff21b
Chunks: 1

--- CHUNK 0 | tok

## Embedding

In [4]:
import math
import ollama


chunk_records = []

for result in results:
    for chunk in result["chunks"]:
        chunk_records.append(
            {
                "doc_id": result["doc_id"],
                "title": result["title"],
                "chunk": chunk,
            }
        )


texts = [
    f"search_document: {record['chunk'].text}"
    for record in chunk_records
]

response = ollama.embed(
    model="nomic-embed-text",
    input=texts,
)

embeddings = response["embeddings"]

print(f"Chunks: {len(chunk_records)}")
print(f"Embeddings: {len(embeddings)}")
print(f"Embedding dimension: {len(embeddings[0])}")

Chunks: 20
Embeddings: 20
Embedding dimension: 768


In [5]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))

    magnitude_a = math.sqrt(
        sum(x * x for x in a)
    )

    magnitude_b = math.sqrt(
        sum(y * y for y in b)
    )

    return dot_product / (magnitude_a * magnitude_b)

In [6]:
query = "growing military cooperation between Saudi Arabia, Turkey, and Pakistan"
queries = [
    "attacks by Houthi forces in Yemen",
    "risk to shipping and energy flows through the Strait of Hormuz",
]

query_response = ollama.embed(
    model="nomic-embed-text",
    input=f"search_query: {query}",
)

query_embedding = query_response["embeddings"][0]

In [7]:
scored_results = []

for record, embedding in zip(chunk_records, embeddings):

    score = cosine_similarity(
        query_embedding,
        embedding,
    )

    scored_results.append(
        {
            "score": score,
            "title": record["title"],
            "doc_id": record["doc_id"],
            "text": record["chunk"].text,
        }
    )


scored_results.sort(
    key=lambda result: result["score"],
    reverse=True,
)

In [8]:
for rank, result in enumerate(scored_results[:5], start=1):

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Similarity: {result['score']:.4f}")
    print(f"Title: {result['title']}")
    print(f"Document ID: {result['doc_id']}")
    print()
    print(result["text"])
    print()

Rank: 1
Similarity: 0.8583
Title: Saudi Arabia, Turkey, Pakistan deepen military cooperation | The Jerusalem Post
Document ID: currents-ba85bb2b-1137-51ad-b536-cc7c0a1db943

saudi arabia, turkey, pakistan deepen military cooperation | the jerusalem post the planned pact would deepen cooperation between riyadh, ankara and islamabad as the three countries seek greater strategic influence in the region.

Rank: 2
Similarity: 0.8397
Title: Saudi Arabia, Turkey and Pakistan set to sign trilateral defence pact
Document ID: currents-e22f3284-0c4f-5d39-a4bc-4aec5446f795

saudi arabia, turkey and pakistan set to sign trilateral defence pact saudi arabia, turkey and pakistan are set to sign a defence agreement in jeddah on friday. the pact could widen military coordination as gulf tensions and regional conflicts intensify.

Rank: 3
Similarity: 0.8224
Title: Why Pakistan, Saudi Arabia, Turkey are signing a new defence pact
Document ID: currents-216725d7-38ca-5d66-860b-f872796fbf14

why pakistan, s

## Chroma DB

In [9]:
import chromadb


CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "argus_document_chunks"

client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME
)

print("Current collection count:", collection.count())

Current collection count: 0


In [10]:
chunk_records = []

for result in results:
    for chunk in result["chunks"]:
        chunk_records.append(
            {
                "chunk_id": chunk.chunk_id,
                "doc_id": result["doc_id"],
                "title": result["title"],
                "text": chunk.text,
            }
        )

texts = [
    f"search_document: {record['text']}"
    for record in chunk_records
]

response = ollama.embed(
    model="nomic-embed-text",
    input=texts,
)

embeddings = response["embeddings"]

print("Chunks:", len(chunk_records))
print("Embeddings:", len(embeddings))

Chunks: 20
Embeddings: 20


In [14]:
collection.upsert(
    ids=[
        record["chunk_id"]
        for record in chunk_records
    ],
    documents=[
        record["text"]
        for record in chunk_records
    ],
    embeddings=embeddings,
    metadatas=[
        {
            "doc_id": record["doc_id"],
            "title": record["title"],
            "provider": "currents",
        }
        for record in chunk_records
    ],
)

print("Persisted collection count:", collection.count())

Persisted collection count: 20


In [19]:
query = "growing military cooperation between Saudi Arabia, Turkey, and Pakistan"
query = "Houthi attacks on Yemeni government forces"

query_response = ollama.embed(
    model="nomic-embed-text",
    input=f"search_query: {query}",
)

query_embedding = query_response["embeddings"][0]

results_chroma = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
)

In [20]:
for rank, (
    doc,
    metadata,
    distance,
) in enumerate(
    zip(
        results_chroma["documents"][0],
        results_chroma["metadatas"][0],
        results_chroma["distances"][0],
    ),
    start=1,
):
    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Distance: {distance:.4f}")
    print(f"Title: {metadata['title']}")
    print(f"Document ID: {metadata['doc_id']}")
    print()
    print(doc)
    print()

Rank: 1
Distance: 0.3650
Title: Houthi missile and drone attacks kill dozens of Yemeni government troops
Document ID: currents-473ab586-3a21-5adc-83f0-393ececff21b

houthi missile and drone attacks kill dozens of yemeni government troops military officials describe the attacks as the deadliest since 2022 as conflict intensifies

Rank: 2
Distance: 0.5988
Title: War on Iran: Saudi Arabia warns of imminent attacks by Iraqi groups and Yemen's Houthis
Document ID: currents-832909a2-159b-55c1-b897-1db1bbbd3a9d

War on Iran: Saudi Arabia warns of imminent attacks by Iraqi groups and Yemen's Houthis

war on iran : saudi arabia warns of imminent attacks by iraqi groups and yemen's houthis the warning follows deadly saudi and us strikes on popular mobilisation forces positions in iraq last month

Rank: 3
Distance: 0.8384
Title: Trump Vows to Jail ‘Leakers’ Over Reports of Munitions Shortage Amid Iran Talks
Document ID: currents-95feb3fb-e3d3-5f66-83e7-393d0436057e

Trump Vows to Jail ‘Leakers’ O